# Project 4 — Optical Bloch Equations
**Topic:** Dynamics of a two-level quantum system driven by a coherent field. The Bloch vector formalism, Rabi oscillations, detuning, and T1/T2 relaxation.


---
## 1. Set Up

### Physical Motivation
The **two-level system** (TLS) is the hydrogen atom of quantum optics. It models any quantum system with two relevant states: ground $|g\rangle$ and excited $|e\rangle$. The density matrix of the TLS is:

$$\hat\rho = \begin{pmatrix} \rho_{gg} & \rho_{ge} \\ \rho_{eg} & \rho_{ee} \end{pmatrix}$$

The **Bloch vector** $\mathbf{B} = (u, v, w)$ is a real 3-component representation:
$$u = \rho_{eg} + \rho_{ge} = 2\,\text{Re}(\rho_{ge}), \quad v = i(\rho_{eg} - \rho_{ge}) = -2\,\text{Im}(\rho_{ge}), \quad w = \rho_{ee} - \rho_{gg}$$

In the **rotating wave approximation (RWA)** and rotating frame at the laser frequency $\omega_L$, the Bloch equations become:

$$\dot{u} = -\delta v - u/T_2$$
$$\dot{v} = \delta u - \Omega_R w - v/T_2$$
$$\dot{w} = \Omega_R v - (w - w_{\text{eq}})/T_1$$

where $\delta = \omega_L - \omega_{eg}$ is the **detuning**, $\Omega_R = \mu E_0/\hbar$ is the **Rabi frequency**, $T_1$ is the **population relaxation time**, and $T_2$ is the **coherence decay time** (dephasing).

### Physical Meaning of $T_1$ and $T_2$
- **$T_1$ (longitudinal relaxation)**: how fast excited population decays ($w \to w_{\text{eq}} = -1$ for a system in ground state at equilibrium). Caused by spontaneous emission, phonon coupling.
- **$T_2$ (transverse relaxation)**: how fast coherence $u, v$ decays. $T_2 \leq 2T_1$ always (pure dephasing can only add to decay). In NMR: $T_2^* \leq T_2$ due to inhomogeneous broadening.

### Bloch Vector Geometry
The Bloch vector lives on or inside the **Bloch sphere**: $u^2 + v^2 + w^2 \leq 1$. Pure states are on the sphere; mixed states are inside. Driving rotates the Bloch vector; relaxation contracts it toward the south pole $w = -1$.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_optical_bloch_sbe")
OUT.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `rk4_step()` — Same as Project 03
See Project 3.1 for full derivation. The key point: we integrate a 3-D ODE system (Bloch vector components), not a 2-D position-velocity system.

### `bloch_rhs(t, y, p)` — Line by Line

```python
u, v, w = y
```
Unpack the Bloch vector: $u$ = in-phase coherence, $v$ = quadrature coherence, $w$ = population inversion ($w = +1$: fully excited; $w = -1$: fully in ground state).

```python
du = -delta * v
dv =  delta * u - omega * w
dw =  omega * v
```
**Coherent (Hamiltonian) evolution only.** These are the Bloch equations with $T_1 = T_2 = \infty$. Physical interpretation:
- $\dot{u} = -\delta v$: detuning rotates the coherence vector in the $u$-$v$ plane at rate $\delta$.
- $\dot{v} = \delta u - \Omega_R w$: driving couples $v$ to the population inversion $w$.
- $\dot{w} = \Omega_R v$: the drive changes population (absorption/emission) through the quadrature component $v$.

On resonance ($\delta = 0$): $u$ is decoupled, and $(v, w)$ undergo **Rabi oscillations** at frequency $\Omega_R$.

```python
if np.isfinite(T2):
    du += -u / T2
    dv += -v / T2
if np.isfinite(T1):
    dw += -(w - w_eq) / T1
```
**Relaxation terms** — added on top of the coherent evolution. `np.isfinite()` allows $T_1 = T_2 = \infty$ (no relaxation, pure coherent dynamics). The $w_{\text{eq}}$ term drives $w$ toward thermal equilibrium (not necessarily $-1$ — at room temperature $w_{\text{eq}} \approx -1$, but at high temperature $w_{\text{eq}} \to 0$).

### `steady_state_absorption_proxy()`
Simulates to long times and reads off $v(t \gg T_2)$ as an absorption proxy. **Physical connection**: in linear response theory, the steady-state absorption spectrum is a **Lorentzian**:

$$\chi''(\delta) \propto \frac{1/T_2}{\delta^2 + 1/T_2^2} \cdot \frac{1}{1 + \Omega_R^2 T_1 T_2}$$

The linewidth is $\Delta\nu = 1/(\pi T_2)$ — the natural linewidth. Strong driving ($\Omega_R \gg 1/\sqrt{T_1 T_2}$) causes **power broadening**: the linewidth grows as $\sqrt{1 + \Omega_R^2 T_1 T_2}$.

### 🔧 Improvement Directions
1. **Bloch sphere 3-D plot**: visualise the trajectory of $(u,v,w)$ on the Bloch sphere. A $\pi$-pulse should flip $w$ from $-1$ to $+1$.
2. **FID and spectrum**: after a $\pi/2$ pulse, let the system evolve freely. The FID signal $s(t) = u(t) + iv(t)$ Fourier-transforms to the absorption lineshape.
3. **Pulse sequence (NMR)**: implement spin echo ($\pi/2 - \tau - \pi - \tau$) and show that inhomogeneous broadening is refocused.
4. **Full density matrix**: go beyond the TLS to a $N$-level system using the Lindblad master equation.

### ⚠️ Known Weaknesses
- **RWA**: neglects counter-rotating terms. Valid only when $\Omega_R \ll \omega_L$ — breaks down for ultrashort pulses.
- **Semiclassical**: treats the electromagnetic field classically. Quantum optics effects (photon statistics, squeezing) require full quantisation.
- **$T_1, T_2$ constant**: in reality, relaxation rates depend on temperature, phonon spectrum, and drive strength.


In [ ]:
def rk4_step(f, t, y, dt, params):
    k1 = f(t,           y,             params)
    k2 = f(t + 0.5*dt,  y + 0.5*dt*k1, params)
    k3 = f(t + 0.5*dt,  y + 0.5*dt*k2, params)
    k4 = f(t + dt,       y + dt*k3,    params)
    return y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def bloch_rhs(t, y, p):
    u, v, w = y
    delta  = p.get("delta",   0.0)      # detuning: omega_laser - omega_transition
    omega  = p.get("omega_R", 1.0)      # Rabi frequency: mu*E0/hbar
    w_eq   = p.get("w_eq",   -1.0)      # equilibrium inversion (thermal)
    T1     = p.get("T1",  np.inf)       # population relaxation
    T2     = p.get("T2",  np.inf)       # coherence dephasing

    # Coherent (Hamiltonian) evolution
    du =  -delta * v
    dv =   delta * u - omega * w
    dw =   omega * v

    # Relaxation (Lindblad terms in Bloch form)
    if np.isfinite(T2):
        du += -u / T2
        dv += -v / T2
    if np.isfinite(T1):
        dw += -(w - w_eq) / T1

    return np.array([du, dv, dw], dtype=float)

def integrate_bloch(params, y0=None, t_min=0.0, t_max=80.0, dt=0.01):
    if y0 is None:
        y0 = np.array([0.0, 0.0, -1.0])   # ground state: w=-1, u=v=0
    t = np.arange(t_min, t_max + 0.5*dt, dt)
    y = np.empty((t.size, 3), dtype=float)
    y[0] = y0
    for j in range(1, t.size):
        y[j] = rk4_step(bloch_rhs, t[j-1], y[j-1], dt, params)
    return t, y

def steady_state_absorption_proxy(delta_values, omega_R=0.05, T1=30.0, T2=10.0,
                                   t_max=250.0, dt=0.05):
    absorption, final_states = [], []
    for delta in delta_values:
        params = {"delta": delta, "omega_R": omega_R, "T1": T1, "T2": T2, "w_eq": -1.0}
        t, y = integrate_bloch(params, t_max=t_max, dt=dt)
        final_mean = y[int(0.9*len(y)):].mean(axis=0)   # time-average last 10%
        absorption.append(final_mean[1])                 # v is absorption proxy
        final_states.append(final_mean)
    return np.array(absorption), np.array(final_states)


---
## 3 & 4. Calculation & Writeouts

### On-resonance Rabi Oscillations ($\delta = 0$, no relaxation)
Starting from ground state $\mathbf{B} = (0, 0, -1)$:
- $u(t) = 0$ (decoupled on resonance)
- $v(t) = \sin(\Omega_R t)$
- $w(t) = -\cos(\Omega_R t)$

Period $= 2\pi/\Omega_R$. The Bloch vector oscillates between $w=-1$ (ground) and $w=+1$ (excited) — these are **Rabi oscillations**.

With relaxation: oscillations are damped. The steady state has $w < -1 + \epsilon$ (population partially inverted by drive, then relaxes).


In [ ]:
# Case 1: On-resonance Rabi oscillations, no relaxation
params_rabi = {"delta": 0.0, "omega_R": 0.5, "T1": np.inf, "T2": np.inf}
t_rabi, y_rabi = integrate_bloch(params_rabi, t_max=50.0, dt=0.02)

print("Rabi period (theory):", 2*np.pi / 0.5, "time units")
print("Bloch vector magnitude (should = 1.0):",
      np.sqrt(y_rabi[:, 0]**2 + y_rabi[:, 1]**2 + y_rabi[:, 2]**2).max())

# Case 2: Detuned oscillations
params_det = {"delta": 0.3, "omega_R": 0.5, "T1": np.inf, "T2": np.inf}
t_det, y_det = integrate_bloch(params_det, t_max=50.0, dt=0.02)
omega_gen = np.sqrt(0.5**2 + 0.3**2)
print(f"\nGeneralised Rabi freq: {omega_gen:.4f} (theory), "
      f"confirms faster oscillation with detuning")

# Case 3: With relaxation
params_relax = {"delta": 0.0, "omega_R": 0.5, "T1": 20.0, "T2": 10.0}
t_relax, y_relax = integrate_bloch(params_relax, t_max=80.0, dt=0.02)

# Case 4: Absorption spectrum
deltas = np.linspace(-2.0, 2.0, 80)
absorb, _ = steady_state_absorption_proxy(deltas, omega_R=0.05, T1=30.0, T2=10.0)
FWHM_numerical = deltas[np.where(absorb > absorb.max()/2)[0][-1]] -                  deltas[np.where(absorb > absorb.max()/2)[0][0]]
FWHM_theory    = 2.0 / 10.0
print(f"\nAbsorption FWHM: numerical={FWHM_numerical:.4f}, theory=2/T2={FWHM_theory:.4f}")


---
## 5. Matplotlib Visualisation

### Key Plots and Their Interpretation
1. **$u,v,w$ vs time (resonant)**: $u=0$, $v$ and $w$ oscillate sinusoidally. $w=+1$ means fully excited (population inverted — this is the laser gain condition).
2. **Damped Rabi oscillations**: envelope decays as $e^{-t/T_2}$. Long-time steady state: $u=v=0$, $w = w_{\text{ss}}$ determined by rate equations.
3. **Absorption spectrum**: Lorentzian centred at $\delta=0$. FWHM $= 2/T_2$. This is the **natural linewidth** — a fundamental quantum limit set by $T_2$.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Resonant Rabi
ax = axes[0,0]
ax.plot(t_rabi, y_rabi[:,0], label='$u$')
ax.plot(t_rabi, y_rabi[:,1], label='$v$')
ax.plot(t_rabi, y_rabi[:,2], label='$w$')
ax.set_title('Rabi oscillations (on resonance, no decay)')
ax.set_xlabel('Time'); ax.legend()

# Detuned vs on-resonance w-component
ax2 = axes[0,1]
ax2.plot(t_rabi, y_rabi[:,2], label=f'$\delta=0$, $\Omega_R=0.5$')
ax2.plot(t_det,  y_det[:,2],  label=f'$\delta=0.3$, $\Omega_R=0.5$', ls='--')
ax2.set_title('Population inversion $w(t)$: detuning effect')
ax2.set_xlabel('Time'); ax2.legend()
ax2.axhline(0, ls=':', color='k', lw=0.8)
ax2.text(1, 0.1, 'Excited state $w>0$', fontsize=8)
ax2.text(1, -0.9, 'Ground state $w<0$', fontsize=8)

# Damped
ax3 = axes[1,0]
ax3.plot(t_relax, y_relax[:,1], label='$v(t)$ (damped)')
ax3.plot(t_relax, y_relax[:,2], label='$w(t)$ (damped)')
envelope = np.exp(-t_relax / 10.0)
ax3.plot(t_relax, envelope,  'k--', lw=1, label='$e^{-t/T_2}$ envelope')
ax3.plot(t_relax, -envelope, 'k--', lw=1)
ax3.set_title(f'Damped Rabi: $T_1=20$, $T_2=10$')
ax3.set_xlabel('Time'); ax3.legend(fontsize=9)

# Absorption spectrum
ax4 = axes[1,1]
ax4.plot(deltas, absorb, 'steelblue', lw=2, label='Numerical')
lor = (1/10.0) / (deltas**2 + (1/10.0)**2)
lor = lor * absorb.max() / lor.max()
ax4.plot(deltas, lor, 'r--', lw=1.5, label=f'Lorentzian (FWHM=2/$T_2$={FWHM_theory:.2f})')
ax4.axvline(0, ls=':', color='k', lw=0.8)
ax4.set_xlabel('Detuning $\delta$')
ax4.set_ylabel('Absorption $\propto v$')
ax4.set_title('Steady-state absorption spectrum')
ax4.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT / "optical_bloch.png", dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Sanity Checks

| Test | Criterion | Physical meaning |
|------|-----------|-----------------|
| $|\mathbf{B}|^2 = 1$ (no relaxation) | $\leq 10^{-6}$ drift | Pure state stays pure |
| $|\mathbf{B}|^2 \leq 1$ (with relaxation) | Always | Mixed states inside sphere |
| Rabi period matches $2\pi/\Omega_R$ | $<0.1\%$ | Correct frequency |
| Absorption peak at $\delta=0$ | Yes | Resonance condition |
| FWHM $\approx 2/T_2$ | $<5\%$ | Natural linewidth |
| $w \to w_{\text{eq}}$ as $t\to\infty$ | Yes | Thermal equilibrium |


In [ ]:
print("=" * 55)
print("SANITY CHECKS — Project 04 Optical Bloch")
print("=" * 55)

# 1. Bloch vector magnitude (no relaxation = pure state)
mag2 = y_rabi[:,0]**2 + y_rabi[:,1]**2 + y_rabi[:,2]**2
drift = np.max(np.abs(mag2 - 1.0))
ok = drift < 1e-5
print(f"\n1. |B|² drift (no relaxation): {drift:.2e}  {'✓' if ok else '✗'}")

# 2. Pure states inside unit sphere (with relaxation)
mag2_r = y_relax[:,0]**2 + y_relax[:,1]**2 + y_relax[:,2]**2
ok2 = np.all(mag2_r <= 1.0 + 1e-8)
print(f"2. |B|² ≤ 1 with relaxation: max={mag2_r.max():.6f}  {'✓' if ok2 else '✗'}")

# 3. Rabi period
v_comp = y_rabi[:, 1]
crossings = np.where(np.diff(np.sign(v_comp)) > 0)[0]
if len(crossings) >= 2:
    T_rabi_num = np.mean(np.diff(t_rabi[crossings]))
    T_rabi_theory = 2*np.pi / 0.5
    ok3 = abs(T_rabi_num - T_rabi_theory) / T_rabi_theory < 0.01
    print(f"3. Rabi period: {T_rabi_num:.4f} (theory: {T_rabi_theory:.4f})  {'✓' if ok3 else '✗'}")

# 4. Absorption peak location
peak_idx = np.argmax(np.abs(absorb))
ok4 = abs(deltas[peak_idx]) < 0.1
print(f"4. Absorption peak at δ = {deltas[peak_idx]:.3f}  (expected ≈ 0)  {'✓' if ok4 else '✗'}")

# 5. Relaxed system reaches equilibrium
w_final = y_relax[-1, 2]
ok5 = abs(w_final - (-1.0)) < 0.1
print(f"5. w(t→∞) = {w_final:.4f}  (expected ≈ -1)  {'✓' if ok5 else '✗'}")

print("\n" + "=" * 55)
